# 04 · What goes with being highly cited

Features read from the papers themselves (timing, data, team, content, form), set against how well each paper is cited **compared with papers posted in the same month**. The sample is the JWST data-use sample (`01_sample.ipynb`): JWST galaxy journal articles and preprints from 2023 on that use JWST data (in ADS's curated JWST bibliography, or a passage of their text scoring at least 7 on the data-use rule), with citations as of the ADS snapshot of this repository's freeze (`common/config.py`).

**What is measured.** The target is each paper's citation percentile within its arXiv-month cohort (0 = least cited of that month, 100 = most cited), which removes the age effect. The headline flag is the top 10% of a cohort.

**What this is not.** These are associations, not causes: a catalogue paper is cited because people use the catalogue, a large team cites itself, a Nature paper was chosen because it looked important. Information that only exists after a paper has been read and cited (read counts, an author's h-index today) is left out, and so is ADS's curated JWST tag. The tag does not depend on citations, but librarians add it months after publication and only to published articles, so it mostly records a paper's age and refereed status rather than its content. Author history is counted only up to the year before publication. Self-citations are counted in the main analysis; section 9 (test 9) repeats it without them.

Sections: 1 configuration · 2 sample · 3 ADS pulls (cached) · 4 features · 5 target · 6 models and the time split · 7 associations · 8 how well the predictions work (training, test, cross-validation) · 9 robustness tests (tree settings, chance, null test, posting-time information, data quality, target choices, linear effects, self-citations).

---
## 1 · Configuration

In [ ]:
import os, sys, re, json, time, html
from collections import Counter, defaultdict
from datetime import date, datetime

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import spearmanr

# Inputs and outputs. The sample comes from 03_topics_sample_titles.ipynb (sample_table.csv: the JWST
# data-use sample, the citations of its topic pull, topic flags and title labels); the ADS caches below
# are filled by section 3 and reused on every later run (data/<freeze>/).
sys.path.insert(0, os.path.abspath(".."))
from common.config import FREEZE, CITES_AS_OF as _CITES_AS_OF, GEN_OUT, GEN_PLOTS, data
from common.config import require_texts
from common.ads import ads_get as _ads_get, bigquery as _bigquery, highlights_for
require_texts()   # abstracts and programme snippets are read below
SAMPLE_TABLE = os.path.join(GEN_OUT, "sample_table.csv")          # one row per sample paper
META_JSON    = data("features_ads_metadata.json")        # cache: per-paper ADS metadata
PROG_JSON    = data("features_programme_snippets.json")  # cache: full-text snippets naming JWST programmes
IDENT_JSON   = data("features_author_identity.json")     # cache: first-author name spellings and histories
SELF_JSON    = data("features_citing_authors.json")      # cache: citing papers and their authors (test 9)
FEATURES_CSV = os.path.join(GEN_OUT, "citation_features.csv")   # the feature table built here
PLOTS_DIR    = GEN_PLOTS

# Analysis settings (see section 5 for how they are used).
CITES_AS_OF    = _CITES_AS_OF   # month of the citation snapshot (common.config: the freeze)
MIN_AGE_MONTHS = 3           # papers posted less than this before the snapshot are too young to model
MIN_COHORT     = 25          # months with fewer papers are pooled with the next month
TEST_FROM      = "2025-04"   # time split: train on cohorts before this month, test on this month and later
REFRESH        = False       # False: reuse the cached ADS pulls
SLEEP_SEC      = 0.3
ADS_BASE       = "https://api.adsabs.harvard.edu/v1"


os.makedirs(PLOTS_DIR, exist_ok=True)

---
## 2 · Sample

The JWST data-use sample (`01_sample.ipynb`: candidate query, data use shown by `bibgroup:JWST` or explicit data use in the
text, journal articles and preprints, galaxy papers), restricted to the papers ADS returned in the topic pull of
`03_topics_sample_titles.ipynb`, whose citation counts are the snapshot used here. Topic flags and the
catchy-title labels come from the same notebook.


In [ ]:
# The sample table of 03_topics_sample_titles.ipynb: one row per sample paper, with the citations of
# its topic pull, a flag per searched topic and the hand label of the title.
m = pd.read_csv(SAMPLE_TABLE)
TOPIC_COLS = [c for c in m.columns if c.startswith("topic: ")]
for c in TOPIC_COLS + ["in_topic_pull"]:
    m[c] = m[c].astype(bool)

# Papers that the topic pull returned (their citation count is this snapshot's). S has one row per
# paper; every feature table below is aligned with it.
S = m[m["in_topic_pull"]].copy().reset_index(drop=True)
S["citations"] = S["citations"].astype(float)
BIBS = list(S["bibcode"])
print(f"{len(S)} papers, {int(S['citations'].sum()):,} citations")


---
## 3 · ADS pulls (cached)

Three pulls, each cached in `data/<freeze>/` and only re-queried for papers not yet in the cache:

1. **Metadata** (bigquery): identifiers (for the arXiv posting month), journal, refereed status, page count, reference list, authors, affiliations, ORCIDs, abstract. arXiv-only preprints have no affiliations, ORCIDs, page counts or references in ADS; those features are missing for them.
2. **Programme snippets**: full-text highlights around "program", "PID", "GO", "GTO", "ERS", ... from which JWST programme IDs are read.
3. **First-author history**: refereed astronomy papers by the first author *before* the publication year, and the year of their first one. Done in three steps:
    - **One identity per person.** Papers share a first author when they carry the same ORCID, or the same surname and full given name (accents, hyphens and middle initials ignored). A name printed with initials only joins a person only if exactly one compatible full-name person exists. Two different ORCIDs are never merged.
    - **Is the name shared?** ADS lists every full name behind "Surname, Initial" with its number of papers. The name is *shared* when other full given names, not compatible with this person's, have at least 3 papers and together at least 5% of this person's own records (Wang B. is Bin, Bo, Bingjie, ...; Schaerer D. is one person).
    - **Count.** One query per person returns their refereed papers per year. Single person: ORCID OR every "Surname, I" spelling, initials-only records included. Shared name: ORCID OR the person's exact full-name spellings only. Records printed with initials only cannot be attributed and are left out, so **for shared names the count is a lower limit**; the feature `fa_shared_name` tells the models which counts these are. **Career start** is the first paper after the last stretch of more than 7 years without any paper, so a stray old record under the same name ("Santini, P." in 1827) does not start a career two centuries early; papers before the career start are not counted either. A shared name with no known full name falls back to ORCID only, or to no count.

    Remaining weak spots, about 2% of people: an identical full name belonging to someone else (a "Charlotte Moore" whose record starts in 1926), and a few names for which ADS's own author synonyms add records.

In [ ]:
# ADS requests through common/ads.py (token from the ADS_TOKEN environment variable).
def ads_get(path, params, retries=3):
    return _ads_get(path, params, retries=retries)


def bigquery(bibcodes, fl):
    """All docs for a list of bibcodes (POST /search/bigquery), paged."""
    return _bigquery("*:*", bibcodes, fl=fl)


# Caches are JSON dicts keyed by bibcode (or by author and year); REFRESH = True ignores them.
def load_cache(path):
    if REFRESH or not os.path.exists(path):
        return {}
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def save_cache(path, cache):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)


# 1. metadata
# Pull 1: metadata for every paper not yet cached, in one bigquery (a search restricted to a list of
# bibcodes). Papers ADS does not return are cached as {} so they are not asked again.
META_FL = "bibcode,identifier,pubdate,pub,property,page_count,reference,author,aff,orcid_pub,abstract,doctype"
meta = load_cache(META_JSON)
missing = [b for b in BIBS if b not in meta]
if missing:
    for d in bigquery(missing, META_FL):
        meta[d["bibcode"]] = d
    for b in missing:
        meta.setdefault(b, {})
    save_cache(META_JSON, meta)
print(f"metadata: {sum(bool(meta.get(b)) for b in BIBS)} of {len(BIBS)} papers ({len(missing)} fetched now)")

# 2. programme snippets
# Pull 2: full-text highlights (short snippets around matching words) from the body and
# acknowledgements, 40 papers per query. hl.maxAnalyzedChars is raised so long papers are searched
# to the end. Partial results are saved every 400 papers.
PROG_HL_Q = "program OR programme OR programs OR programmes OR PID OR GO OR GTO OR ERS OR ERO OR DDT OR proposal"
prog_hl = load_cache(PROG_JSON)
missing = [b for b in BIBS if b not in prog_hl]
for i in range(0, len(missing), 40):
    part = missing[i:i + 40]
    prog_hl.update(highlights_for(part, {"hl.fl": "body,ack", "hl.q": PROG_HL_Q, "hl.snippets": "10",
                                         "hl.maxAnalyzedChars": "5000000"}))   # asked by identifier
    for b in part:
        prog_hl.setdefault(b, {})
    if i % 400 == 0:
        save_cache(PROG_JSON, prog_hl)
    time.sleep(SLEEP_SEC)
if missing:
    save_cache(PROG_JSON, prog_hl)
print(f"programme snippets: {sum(bool(prog_hl.get(b)) for b in BIBS)} of {len(BIBS)} papers have some "
      f"({len(missing)} queried now)")

In [ ]:
# 3. first-author history, up to the year before publication
# Three steps: (a) decide which papers share a first author (one identity per person across the
# sample); (b) ask ADS whether that person's 'Surname, Initial' is shared with other people; (c) count
# the person's earlier refereed papers with a query that fits (b). See the markdown above for why.
import unicodedata

MIN_OTHER = 3      # another full given name with at least this many ADS papers counts as another person ...
MIN_SHARE = 0.05   # ... but only if all such people together have >= 5% of this person's own records
HIST_FQ = ["property:refereed", "collection:astronomy"]


def deaccent(s):
    """'Pérez-González' -> 'Perez-Gonzalez'."""
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))


def parse_name(name):
    """'Pérez-González, Pablo G.' -> ('Perez-Gonzalez', 'P', ('pablo',), ('p', 'g')): surname without
    accents, first initial, the full given-name tokens (lower case, hyphens dropped so 'Jun-Chao' =
    'Junchao'; initials like 'G.' or 'J.-C.' left out) and every initial."""
    sur, _, giv = deaccent(name).partition(",")
    toks = [t for t in re.split(r"[\s.]+", giv.strip()) if t]
    full = tuple(t.replace("-", "").lower() for t in toks
                 if len(t.replace("-", "")) > 1 and not re.fullmatch(r"([A-Z]-?)+", t))
    return sur.strip(), (toks[0][0].upper() if toks else ""), full, tuple(t[0].lower() for t in toks)


def compatible(a, b):
    """Can full given names a and b (token tuples) be the same person? Equal, or one continues the other
    at a token boundary: 'daniel' / 'daniel olivier' and 'junchao' / 'jun chao' yes, 'bing' / 'bingjie' no."""
    if not a or not b:
        return True
    sa, sb = "".join(a), "".join(b)
    if sa == sb:
        return True
    short, long_ = (sa, b) if len(sa) < len(sb) else (sb, a)
    acc = ""
    for t in long_:                      # build the longer name token by token; it must hit the shorter one
        acc += t
        if acc == short:
            return True
    return False


# ── (a) one identity per person in the sample ──
# Union-find over papers: papers are joined when they share the first author's ORCID, or have the same
# surname and compatible full given names; a name with initials only ('Faisst, A. L.') joins a person
# only when exactly one compatible full-name person exists. Two different ORCIDs are never joined.
FA = []
for b, y in zip(S["bibcode"], S["year"]):
    d = meta.get(b) or {}
    au = d.get("author") or []
    orc = (d.get("orcid_pub") or ["-"])[0] if au else None
    sur, ini, full, inits = parse_name(au[0]) if au else ("", "", (), ())
    FA.append(dict(bibcode=b, year=int(y), name=au[0] if au else None, orcid=orc if orc and orc != "-" else None,
                   sur=sur, ini=ini, full=full, inits=inits))
parent = list(range(len(FA)))
orcids_of = [{p["orcid"]} - {None} for p in FA]     # ORCIDs held by each group (kept at the group's root)


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


def union(i, j):
    ri, rj = find(i), find(j)
    if ri == rj or (orcids_of[ri] and orcids_of[rj] and orcids_of[ri] != orcids_of[rj]):
        return                           # already one group, or two different ORCIDs = two people
    parent[rj] = ri
    orcids_of[ri] |= orcids_of[rj]


by_orcid, by_si = defaultdict(list), defaultdict(list)
for i, p in enumerate(FA):
    if p["orcid"]:
        by_orcid[p["orcid"]].append(i)
    if p["name"]:
        by_si[(p["sur"].lower(), p["ini"])].append(i)
for idx in by_orcid.values():                                    # same ORCID
    for j in idx[1:]:
        union(idx[0], j)
for idx in by_si.values():                                       # same surname, compatible full given names
    fulls = [i for i in idx if FA[i]["full"]]
    for k, i in enumerate(fulls):
        for j in fulls[k + 1:]:
            if compatible(FA[i]["full"], FA[j]["full"]):
                union(i, j)
for idx in by_si.values():                                       # initials only: the one compatible person
    for i in (i for i in idx if not FA[i]["full"]):
        # a second initial, when both names have one, must agree ('A. J.' never joins 'Andreas L.')
        cands = {find(j) for j in idx if FA[j]["full"] and (len(FA[i]["inits"]) < 2 or len(FA[j]["inits"]) < 2
                                                            or FA[i]["inits"][1] == FA[j]["inits"][1])}
        if len(cands) == 1:
            union(cands.pop(), i)
FA = pd.DataFrame(FA, index=S.index)
FA["person"] = [find(i) for i in range(len(FA))]
FA.loc[FA["name"].isna(), "person"] = -1                         # no author list: no person


# ── (b) and (c): ADS ──
# ADS GET that also rides out short network outages (waits 30 s, 1 min, 2 min before giving up).
def ads_get_patient(path, params):
    for wait in (30, 60, 120, None):
        try:
            return ads_get(path, params)
        except requests.exceptions.ConnectionError:
            if wait is None:
                raise
            time.sleep(wait)


# Facet query: every spelling ADS has under 'Surname, I' (refereed astronomy), with its number of papers.
def ads_forms(q, sur, ini, limit=500):
    resp = ads_get_patient("/search/query", {"q": q, "fq": HIST_FQ, "rows": 0, "facet": "true",
                                     "facet.field": "author_facet_hier", "facet.prefix": f"1/{sur}, {ini}/",
                                     "facet.limit": limit, "facet.mincount": 1})
    time.sleep(SLEEP_SEC)
    f = resp["facet_counts"]["facet_fields"]["author_facet_hier"]
    return {k.split("/", 2)[2]: v for k, v in zip(f[::2], f[1::2])}


def given_groups(forms):
    """Spellings with a full given name, grouped by that name: {('bingjie',): {'Wang, Bingjie': 86, ...}}."""
    out = defaultdict(dict)
    for k, v in forms.items():
        if parse_name(k)[2]:
            out[parse_name(k)[2]][k] = v
    return out


# A person's refereed papers per publication year (one query, 'year' facet): the count before any year
# and the year of the first paper both follow from it.
def year_counts(q):
    resp = ads_get_patient("/search/query", {"q": q, "fq": HIST_FQ, "rows": 0, "facet": "true",
                                             "facet.field": "year", "facet.limit": 200, "facet.mincount": 1})
    time.sleep(SLEEP_SEC)
    f = resp["facet_counts"]["facet_fields"]["year"]
    return {int(y): n for y, n in zip(f[::2], f[1::2])}


CAREER_GAP = 7    # years: a longer stretch between two papers ends the search for the career start (checked by
                  # hand on 100 people, general/tools/author_check.py: 5 years cuts real careers)


def career_start(years):
    """First year of the person's career: walk back from the most recent paper and stop at the first
    stretch of more than CAREER_GAP years between two papers. A stray old record under the same name
    ('Santini, P.' in 1827) is cut off this way; a continuous record since 1973 is kept whole."""
    ys = sorted(years, reverse=True)
    start = ys[0] if ys else None
    for newer, older in zip(ys, ys[1:]):
        if newer - older > CAREER_GAP:
            break
        start = older
    return start


# Earlier refereed papers matching a query, from the start of the career (career_start on the same papers) up to
# the year before `year`: papers before the career start are a namesake's and are not counted.
def history(q, year, cache):
    if f"years|{q}" not in cache:
        cache[f"years|{q}"] = {str(y): n for y, n in year_counts(q).items()}
    before = {int(y): n for y, n in cache[f"years|{q}"].items() if int(y) < year}
    first = career_start(before)
    return {"n": sum(n for y, n in before.items() if y >= first) if before else 0, "first": first}


def lookup_rule(g, cache):
    """The ADS query for one person (rows g of FA) and its rule:
    'single'   no one else shares 'Surname, I' -> ORCID OR author:"Surname, I" (every spelling, initials too);
    'shared'   others share it -> ORCID OR this person's exact full-name spellings (=author: no initials-only
               records, so the count is a LOWER LIMIT);
    'shared-orcid' shared, no full name known for this person -> ORCID only (lower limit);
    'unresolved'   shared, no full name, no ORCID -> no count."""
    sur, ini = g["sur"].iloc[0], g["ini"].iloc[0]
    orcs = sorted({o for o in g["orcid"] if o})
    orc_q = [f'orcid:"{o}"' for o in orcs]
    key = f"forms|{sur}, {ini}"
    if key not in cache:
        cache[key] = ads_forms(f'author:"{sur}, {ini}"', sur, ini)
    groups = given_groups(cache[key])
    mine_full = {f for f in g["full"] if f}
    if not mine_full and orcs:
        # initials only in the sample: take the full given name that dominates the person's ORCID-tagged
        # papers (co-authors with the same surname and initial appear there too, hence 'dominates')
        okey = f"orcidforms|{'|'.join(orcs)}|{sur}, {ini}"
        if okey not in cache:
            cache[okey] = ads_forms(" OR ".join(orc_q), sur, ini, limit=50)
        ranked = sorted(((k, sum(v.values())) for k, v in given_groups(cache[okey]).items()), key=lambda kv: -kv[1])
        if ranked and (len(ranked) == 1 or ranked[0][1] >= 2 * ranked[1][1]):
            mine_full = {ranked[0][0]}
    if mine_full:
        mine = {k: v for gname, d in groups.items() if any(compatible(gname, f) for f in mine_full) for k, v in d.items()}
        others = {gname: sum(d.values()) for gname, d in groups.items()
                  if not any(compatible(gname, f) for f in mine_full) and sum(d.values()) >= MIN_OTHER}
        if sum(others.values()) < MIN_SHARE * sum(mine.values()):
            others = {}                  # namesakes too small to matter
    else:
        # initials only and no ORCID name: a single full given name in ADS is taken to be this person
        big = {gname: d for gname, d in groups.items() if sum(d.values()) >= MIN_OTHER}
        mine = next(iter(big.values())) if len(big) == 1 else {}
        others = {gname: sum(d.values()) for gname, d in big.items()} if len(big) > 1 else {}
    if not others:
        return "single", " OR ".join([f'author:"{sur}, {ini}"'] + orc_q)
    if mine:
        return "shared", " OR ".join([f'=author:"{k}"' for k in sorted(mine)] + orc_q)
    if orcs:
        return "shared-orcid", " OR ".join(orc_q)
    return "unresolved", None


# One rule per person, one history query per person; everything cached in IDENT_JSON and saved
# every 100 people, so an interrupted run resumes where it stopped.
ident = load_cache(IDENT_JSON)
rule, query, n_prior, first_year = {}, {}, {}, {}
people = [(pid, g) for pid, g in FA[FA["person"] >= 0].groupby("person")]
n_before = len(ident)
for k, (pid, g) in enumerate(people):
    r, q = lookup_rule(g, ident)
    for i in g.index:
        rule[i], query[i] = r, q
    for y in sorted(set(g["year"])):
        if q is None:
            continue
        h = history(q, y, ident)
        for i in g.index[g["year"] == y]:
            n_prior[i], first_year[i] = h["n"], h["first"]
    if k % 100 == 99:
        save_cache(IDENT_JSON, ident)
        print(f"  people: {k + 1} of {len(people)}", end="\r")
save_cache(IDENT_JSON, ident)
FA["rule"] = pd.Series(rule)
FA["n_prior"] = pd.Series(n_prior, dtype=float)
FA["first_year"] = pd.Series(first_year, dtype=float)
# The record of every paper's first author, read by the hand check of 100 people (general/tools/author_check.py).
FA.assign(query=pd.Series(query)).loc[:, ["bibcode", "year", "name", "orcid", "person", "rule", "query", "n_prior",
                                          "first_year"]].to_csv(os.path.join(GEN_OUT, "first_author_records.csv"),
                                                                index=False)

# Check: for one person the earlier-paper count can only grow with the publication year.
drops = sum((g.sort_values("year")["n_prior"].diff() < 0).any() for _, g in FA[FA["person"] >= 0].groupby("person"))
print(f"first authors: {len(FA):,} papers -> {FA.loc[FA['person'] >= 0, 'person'].nunique():,} people "
      f"({len(ident) - n_before} ADS queries now); counts that drop in a later year: {drops}")
display(FA["rule"].value_counts().rename("papers").to_frame())

---
## 4 · Features

Five groups of features.

| Group | Features |
|---|---|
| **Timing** | how early the paper came on its topic (rank of its arXiv date among the papers on the same topic, 0 = first); papers on that topic already out |
| **Data** | JWST programmes named in the text (count, newest cycle, data age); uses public zero-exclusive-access data (ERO, ERS, COSMOS-Web, PRIMER, UNCOVER, NGDEEP, FRESCO, JEMS); uses GTO data (JADES, PEARLS, CANUCS, MIDIS); instruments and modes named in the abstract; other facilities (ALMA, HST) |
| **Team** | number of authors (log), large collaboration (≥ 50), countries and continents, first author's region, first author's earlier refereed papers and career length, whether the first author's name is shared in ADS |
| **Content** | the nine topics of the topic list (`common/trending_shared.py`); catalogue / data-release / survey-overview paper; instrument or calibration paper; abstract wording (claims of firsts and surprises, very high redshift, hedging), length and amount of numbers |
| **Form** | refereed, journal (Nature / Science family, ApJL, ApJ, A&A, MNRAS, other, preprint only), Letter, pages, references, open data or code mentioned; title length, separator, question, survey name, catchy (borderline counts as not catchy) |

Programme cycles come from programme numbers (Cycle 1: 1000-2999, Cycle 2: 3000-4999, Cycle 3: 5000-6999, Cycle 4: 7000+), an approximation; the public and GTO lists name the main galaxy programmes only.

In [ ]:
# ── timing: arXiv posting month (identifier), else publication month ──
# Month the paper was first posted, read from its arXiv identifier (new style 'arXiv:2309.01234' or
# old-style bibcode '2023arXiv230901234X'); papers never on arXiv fall back to the publication
# month.
def arxiv_month(b):
    d = meta.get(b) or {}
    for i in d.get("identifier") or []:
        mm = re.match(r"arXiv:(\d{2})(\d{2})\.\d{4,5}", i)
        if mm:
            return f"20{mm.group(1)}-{mm.group(2)}"
        mm = re.match(r"(\d{4})arXiv(\d{2})(\d{2})", i)
        if mm:
            return f"20{mm.group(2)}-{mm.group(3)}"
    pd_ = (d.get("pubdate") or "")[:7]
    return pd_.replace("-00", "-01") if pd_ else None


# Posting month as text ('2024-03', used to group papers) and as a date (used for ordering).
S["month"] = S["bibcode"].map(arxiv_month)
S["month_date"] = pd.to_datetime(S["month"], format="%Y-%m", errors="coerce")

# first mover: rank of the posting month among the papers on the same topic (0 = first, 1 = last); a
# paper on several topics takes its earliest rank. Ties share the average rank.
# first_mover: the paper's position in time among papers on the same topic (0 = earliest, 1 =
# latest). topic_before: how many papers on that topic were already out. np.fmin keeps the smallest
# value over the paper's topics and ignores NaN.
first_mover = pd.Series(np.nan, index=S.index)
topic_before = pd.Series(np.nan, index=S.index)
for t in TOPIC_COLS:
    on = S[t] & S["month_date"].notna()
    r = S.loc[on, "month_date"].rank(pct=True, method="average")
    first_mover.loc[on] = np.fmin(first_mover.loc[on], r)
    before = S.loc[on, "month_date"].map(lambda x, s=S.loc[on, "month_date"]: int((s < x).sum()))
    topic_before.loc[on] = np.fmin(topic_before.loc[on], before)

# F collects the features (one column each), GROUPS records which group each belongs to; add() does
# both.
F = pd.DataFrame(index=S.index)
GROUPS = defaultdict(list)


def add(group, name, values):
    F[name] = values
    GROUPS[group].append(name)


add("Timing", "topic_rank_earliest", first_mover)
add("Timing", "log_topic_papers_before", np.log10(1 + topic_before))

In [ ]:
# ── data ──
# Hand-made lookup tables: when each JWST cycle started observing, and the main galaxy programmes
# whose data were public from the start (PUBLIC) or belong to instrument teams (GTO).
CYCLE_START = {1: 2022.5, 2: 2023.5, 3: 2024.5, 4: 2025.5}   # approximate start of each cycle's observations
PUBLIC = {**{p: "ERO" for p in range(2731, 2740)},
          **{p: "ERS" for p in (1288, 1309, 1324, 1328, 1334, 1335, 1345, 1355, 1364, 1366, 1373, 1386)},
          1727: "COSMOS-Web", 1837: "PRIMER", 2561: "UNCOVER", 2079: "NGDEEP", 1895: "FRESCO", 1963: "JEMS"}
GTO = {1180: "JADES", 1181: "JADES", 1210: "JADES", 1286: "JADES", 1287: "JADES", 3215: "JADES",
       1176: "PEARLS", 1208: "CANUCS", 1283: "MIDIS"}
# A programme word ('program', 'PID', 'GO', '#', ...) and the 4-digit numbers that follow it,
# optionally written as 'JW01234'.
PROG_WORD = re.compile(r"\b(?:program(?:me)?s?|PIDs?|proposals?|GO|GTO|ERS|ERO|DDT?|IDs?)\b|#", re.I)
NUM = re.compile(r"(?<![\d.])(?:JW0?)?(\d{4})(?![\d.])")


def programmes(b):
    """JWST programme IDs named within 80 characters after a programme word."""
    out = set()
    for texts in (prog_hl.get(b) or {}).values():
        for t in texts if isinstance(texts, list) else []:
            t = html.unescape(re.sub(r"</?em>", "", t))
            for mm in PROG_WORD.finditer(t):
                for n in NUM.findall(t[mm.end(): mm.end() + 80]):
                    n = int(n)
                    if 1000 <= n <= 9999 and not 2015 <= n <= 2030:   # not a year
                        out.add(n)
    return out


# Approximate JWST cycle from the programme number.
def cycle(p):
    return 1 if p < 3000 else 2 if p < 5000 else 3 if p < 7000 else 4


# Per paper: the set of programme IDs, the posting date as a decimal year, the newest cycle used,
# and from them the data features (data age = posting date minus the start of that cycle).
progs = S["bibcode"].map(programmes)
year_frac = S["month_date"].dt.year + (S["month_date"].dt.month - 0.5) / 12
newest = progs.map(lambda ps: max(map(cycle, ps)) if ps else np.nan)
add("Data", "n_programmes", progs.map(len).astype(float))
add("Data", "newest_cycle", newest)
add("Data", "data_age_years", year_frac - newest.map(CYCLE_START))
add("Data", "public_survey_data", progs.map(lambda ps: float(any(p in PUBLIC for p in ps))))
add("Data", "gto_data", progs.map(lambda ps: float(any(p in GTO for p in ps))))

# Instruments, modes and other facilities named in the abstract: 1 if named, else 0.
abstract = S["bibcode"].map(lambda b: html.unescape((meta.get(b) or {}).get("abstract") or ""))
for name, pat in [("nircam", r"NIRCam"), ("nirspec", r"NIRSpec"), ("miri", r"\bMIRI\b"), ("niriss", r"NIRISS"),
                  ("prism", r"\bprism\b"), ("ifu", r"\bIFU\b|integral[- ]field"), ("grism", r"\bgrism"),
                  ("alma", r"\bALMA\b"), ("hst", r"\bHST\b|Hubble Space")]:
    add("Data", f"abs_{name}", abstract.str.contains(pat, case=False, regex=True).astype(float))

In [ ]:
# ── team ──
# Country names (and common spellings) found at the end of affiliation strings, with the region each
# belongs to. US affiliations often give only a state and ZIP code; US_STATE catches those.
COUNTRIES = {   # alias -> (country, region); matched in the last two comma-separated parts of an affiliation
    "USA": ("USA", "N. America"), "United States": ("USA", "N. America"), "U.S.A": ("USA", "N. America"),
    "Canada": ("Canada", "N. America"), "Mexico": ("Mexico", "N. America"),
    "UK": ("UK", "Europe"), "United Kingdom": ("UK", "Europe"), "England": ("UK", "Europe"), "Scotland": ("UK", "Europe"),
    "Germany": ("Germany", "Europe"), "Deutschland": ("Germany", "Europe"), "France": ("France", "Europe"),
    "Italy": ("Italy", "Europe"), "Italia": ("Italy", "Europe"), "Spain": ("Spain", "Europe"), "España": ("Spain", "Europe"),
    "Netherlands": ("Netherlands", "Europe"), "The Netherlands": ("Netherlands", "Europe"),
    "Denmark": ("Denmark", "Europe"), "Sweden": ("Sweden", "Europe"), "Switzerland": ("Switzerland", "Europe"),
    "Austria": ("Austria", "Europe"), "Belgium": ("Belgium", "Europe"), "Portugal": ("Portugal", "Europe"),
    "Ireland": ("Ireland", "Europe"), "Finland": ("Finland", "Europe"), "Norway": ("Norway", "Europe"),
    "Poland": ("Poland", "Europe"), "Czech Republic": ("Czechia", "Europe"), "Czechia": ("Czechia", "Europe"),
    "Greece": ("Greece", "Europe"), "Estonia": ("Estonia", "Europe"), "Hungary": ("Hungary", "Europe"),
    "Israel": ("Israel", "Asia"), "Turkey": ("Turkey", "Asia"), "China": ("China", "Asia"), "P. R. China": ("China", "Asia"),
    "PR China": ("China", "Asia"), "Japan": ("Japan", "Asia"), "Korea": ("Korea", "Asia"), "Republic of Korea": ("Korea", "Asia"),
    "Taiwan": ("Taiwan", "Asia"), "India": ("India", "Asia"), "Hong Kong": ("China", "Asia"), "Singapore": ("Singapore", "Asia"),
    "Iran": ("Iran", "Asia"), "Thailand": ("Thailand", "Asia"),
    "Australia": ("Australia", "Oceania"), "New Zealand": ("New Zealand", "Oceania"),
    "Chile": ("Chile", "S. America"), "Brazil": ("Brazil", "S. America"), "Argentina": ("Argentina", "S. America"),
    "Colombia": ("Colombia", "S. America"), "South Africa": ("South Africa", "Africa"),
}
US_STATE = re.compile(r"\b(?:AL|AZ|CA|CO|CT|DC|FL|GA|HI|IL|IN|MA|MD|MI|MN|NC|NH|NJ|NM|NY|OH|OR|PA|RI|TN|TX|UT|VA|WA|WI)\s+\d{5}\b")


def countries_of(aff):
    """Countries named in one author's affiliation string (several affiliations are ';'-separated)."""
    out = []
    for one in (aff or "").split(";"):
        parts = [p.strip().rstrip(".") for p in one.split(",") if p.strip()]
        hit = None
        for p in parts[-2:][::-1]:
            if p in COUNTRIES:
                hit = COUNTRIES[p]
                break
        if not hit and US_STATE.search(one):
            hit = COUNTRIES["USA"]
        if hit:
            out.append(hit)
    return out


# Per paper: number of different countries and regions over all authors, and the region of the first
# author (missing for preprints, which have no affiliations in ADS).
def team(b):
    d = meta.get(b) or {}
    affs = [a for a in (d.get("aff") or []) if a and a != "-"]
    if not affs:
        return dict(n_countries=np.nan, n_regions=np.nan, fa_region=None)
    per_author = [countries_of(a) for a in affs]
    cs = {c for pa in per_author for c, _ in pa}
    rs = {r for pa in per_author for _, r in pa}
    fa = per_author[0][0][1] if (d.get("aff") or ["-"])[0] != "-" and per_author[0] else None
    return dict(n_countries=float(len(cs)) if cs else np.nan, n_regions=float(len(rs)) if rs else np.nan, fa_region=fa)


# Team features. Author count is log-scaled because its effect is closer to proportional (10 -> 20
# authors matters like 50 -> 100); missing values stay NaN.
T = pd.DataFrame([team(b) for b in S["bibcode"]], index=S.index)
n_auth = S["bibcode"].map(lambda b: len((meta.get(b) or {}).get("author") or [])).replace(0, np.nan)
add("Team", "log_n_authors", np.log10(n_auth))
add("Team", "large_collab", (n_auth >= 50).astype(float).where(n_auth.notna()))
add("Team", "n_countries", T["n_countries"])
add("Team", "n_regions", T["n_regions"])
for reg in ("Europe", "N. America", "Asia"):
    add("Team", f"fa_{reg.lower().replace('. ', '_')}", (T["fa_region"] == reg).astype(float).where(T["fa_region"].notna()))
# First author's history before the publication year (section 3): number of refereed papers (log),
# years since their first one, and whether their name is shared with other people in ADS (then the
# count is a lower limit). No count (no author list, or a shared name that cannot be resolved) = NaN.
add("Team", "fa_log_prior_papers", np.log10(1 + FA["n_prior"]))
add("Team", "fa_career_years", (S["year"] - FA["first_year"]).clip(lower=0))
add("Team", "fa_shared_name", FA["rule"].map({"single": 0.0, "shared": 1.0, "shared-orcid": 1.0}))

In [ ]:
# ── content ──
# Clean title (HTML tags removed) and title + abstract, the text the content features search.
title = S["title"].fillna("").map(lambda t: html.unescape(re.sub(r"<[^>]+>", "", t)))
text = title + " " + abstract
# The nine topics of the topic list (common/trending_shared.py), as 0/1 features.
for t in TOPIC_COLS:
    add("Content", "topic_" + re.sub(r"\W+", "_", t[7:].replace(" (excl. LRDs)", "").lower()).strip("_"), S[t].astype(float))
add("Content", "catalogue_or_release", text.str.contains(
    r"\bcatalog(?:ue)?s?\b|data release|\bDR\d|survey (?:design|overview|description)|public release|"
    r"we (?:release|present) (?:the |a )?(?:public )?(?:catalog|mosaic|data product)", case=False).astype(float))
add("Content", "instrument_paper", title.str.contains(
    r"commissioning|in-flight|performance|calibration|pipeline|data reduction", case=False).astype(float))
# Wording of the abstract: words claiming novelty or surprise (count), a redshift of about 10 or
# more, hedging words (count), length (log words), and how many numbers it quotes per 100 words.
add("Content", "abs_claims", abstract.str.count(
    r"(?i)\bfirst\b|\bdiscover|unprecedented|surpris|unexpected|puzzl|tension|challeng|record").astype(float))
add("Content", "abs_very_high_z", abstract.str.contains(
    r"z\s*(?:[>≳~∼≈=]|\\gtrsim|\\sim)\s*(?:1[0-9]|9\.\d)|redshifts? (?:above|beyond|of) 1[0-9]", case=False).astype(float))
add("Content", "abs_hedging", abstract.str.count(r"(?i)\bmay\b|\bmight\b|\bsuggest|tentative|possibl|candidate").astype(float))
add("Content", "abs_log_words", np.log10(1 + abstract.str.split().map(len)))
add("Content", "abs_numbers_per_100w", 100 * abstract.str.count(r"\d+(?:\.\d+)?") / abstract.str.split().map(len).clip(lower=1))

In [ ]:
# ── form ──
# Journal categories, matched on ADS's publication name; anything else refereed counts as 'other
# journal', and unrefereed papers are preprints.
JOURNALS = [("nature_science", r"^Nature|^Science"), ("apjl", r"Astrophysical Journal Letters"),
            ("apj", r"^The Astrophysical Journal$|Astrophysical Journal Supplement"),
            ("aa", r"Astronomy and Astrophysics|Astronomy & Astrophysics"), ("mnras", r"Monthly Notices")]
pub = S["bibcode"].map(lambda b: (meta.get(b) or {}).get("pub") or "")
prop = S["bibcode"].map(lambda b: set((meta.get(b) or {}).get("property") or []))
refereed = prop.map(lambda p: "REFEREED" in p)
add("Form", "refereed", refereed.astype(float))
known = pd.Series(False, index=S.index)
for name, pat in JOURNALS:
    hit = pub.str.contains(pat, regex=True)
    add("Form", f"journal_{name}", hit.astype(float))
    known |= hit
add("Form", "journal_other", (refereed & ~known).astype(float))
# Letters carry an 'L' before the page number in their bibcode (e.g. 2023ApJ...942L..14R).
add("Form", "letter", S["bibcode"].str.match(r"\d{4}[A-Za-z&.]+?\.*\d+L").astype(float))
add("Form", "log_pages", np.log10(S["bibcode"].map(lambda b: (meta.get(b) or {}).get("page_count")).astype(float)))
add("Form", "log_references", np.log10(1 + S["bibcode"].map(lambda b: len((meta.get(b) or {}).get("reference") or []))
                                       .astype(float).replace(0, np.nan)))
add("Form", "open_data_code", abstract.str.contains(
    r"github|zenodo|publicly available|we (?:release|make)\b|made (?:publicly )?available|is available at", case=False).astype(float))
# Title features: length in words, a separator (':' or a dash), a question mark, a survey name, and
# the hand label from the catchy-title study.
add("Form", "title_words", title.str.split().map(len).astype(float))
add("Form", "title_separator", title.str.contains(r":|\s[—–-]\s").astype(float))
add("Form", "title_question", title.str.contains(r"\?").astype(float))
add("Form", "title_survey_name", title.str.contains(
    r"\b(?:JADES|CEERS|UNCOVER|PRIMER|COSMOS-Web|COSMOS-3D|NGDEEP|FRESCO|GLASS|PEARLS|CANUCS|RUBIES|EXCELS|AURORA|"
    r"MegaScience|ASPIRE|EIGER|SMILES|JEMS|MIDIS|GA-NIFS|BlueDOGs|CAPERS|ALT|SAPPHIRES|Technicolor|MINERVA)\b").astype(float))
add("Form", "title_catchy", S["catchy_label"].eq("catchy").astype(float))   # borderline counts as not catchy

# All features as floats (yes/no as 1/0, missing as NaN), and the share of papers that have each
# feature.
F = F.astype(float)
FEATURES = list(F.columns)
cov = F.notna().mean().mul(100).round(0)
print(f"{len(FEATURES)} features in {len(GROUPS)} groups: " + ", ".join(f"{g} {len(v)}" for g, v in GROUPS.items()))
print("Coverage below 100%:", {k: f"{v:.0f}%" for k, v in cov[cov < 100].items()})

---
## 5 · Target: citation percentile within the posting month

Citations depend strongly on age, so papers are compared only with papers posted in the same month (months with fewer than `MIN_COHORT` papers are pooled with the next month). The percentile is 0-100 within that cohort; **top 10%** = percentile ≥ 90. Papers posted within `MIN_AGE_MONTHS` of the snapshot are too young to tell apart and are left out of the models.

In [ ]:
# cohorts: calendar months, small ones pooled forward
# Cohorts: walk through the months in order; a new cohort starts only once the current one has
# MIN_COHORT papers, so small months join the following ones.
months = sorted(S["month"].dropna().unique())
cohort_of, current, size = {}, None, 0
counts = S["month"].value_counts()
for mo in months:
    if current is None or size >= MIN_COHORT:
        current, size = mo, 0
    cohort_of[mo] = current
    size += counts[mo]
S["cohort"] = S["month"].map(cohort_of)
# Only papers posted at least MIN_AGE_MONTHS before the citation snapshot are modelled. Their
# percentile is the rank of their citations within the cohort (ties share the average rank), scaled
# to 0-100.
cut = (pd.Period(CITES_AS_OF, "M") - MIN_AGE_MONTHS).strftime("%Y-%m")
S["modelled"] = S["month"].notna() & (S["month"] < cut)
S["pct"] = np.nan
S.loc[S["modelled"], "pct"] = (S[S["modelled"]].groupby("cohort")["citations"]
                               .rank(pct=True, method="average").mul(100))
S["top10"] = S["pct"] >= 90

M = S["modelled"]
print(f"{M.sum()} papers modelled (posted before {cut}), {S.loc[M, 'cohort'].nunique()} cohorts "
      f"(median {S[M].groupby('cohort').size().median():.0f} papers); top 10%: {S.loc[M, 'top10'].sum()}")

# Save the feature table: identifiers, target and all features, one row per paper.
out = pd.concat([S[["bibcode", "title", "year", "month", "cohort", "citations", "pct", "top10"]], F], axis=1)
out.to_csv(FEATURES_CSV, index=False)
print(f"Feature table -> {FEATURES_CSV}")

---
## 6 · Models and the time split

Two models predict the percentile: a ridge regression (standardised features, missing values imputed with the median plus a missing-value indicator) and gradient-boosted trees (which handle missing values and non-linear effects themselves). The split is in time: trained on cohorts before `TEST_FROM`, tested on the later ones, so nothing from the test period leaks into training.

Scores on the test set: **Spearman ρ** between predicted and actual percentile, and **AUC** for picking the top 10% (0.5 = chance).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score

# D: the modelled papers; X: their features; y: their citation percentile. train / test: the time
# split at TEST_FROM.
D = S[M].reset_index(drop=True)
X = F[M].reset_index(drop=True)
y = D["pct"].to_numpy()
train = (D["cohort"] < TEST_FROM).to_numpy()
test = ~train
print(f"train: {train.sum()} papers (cohorts {D.loc[train, 'cohort'].min()} to {D.loc[train, 'cohort'].max()}); "
      f"test: {test.sum()} ({D.loc[test, 'cohort'].min()} to {D.loc[test, 'cohort'].max()})")


# Linear model: missing values replaced by the median (plus a 0/1 column saying a value was
# missing), features standardised, ridge regression with the penalty chosen by internal cross-
# validation (efficient leave-one-out) among RIDGE_ALPHAS.
RIDGE_ALPHAS = np.logspace(-1, 4, 30)   # 30 penalty strengths, 0.1 to 10,000, evenly spaced in log


def ridge():
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                         RidgeCV(alphas=RIDGE_ALPHAS))


def alpha_at_edge(a):
    """True if the chosen penalty is the smallest or largest on the grid: then the best value may lie
    outside it, and the grid should be widened."""
    return np.isclose(a, RIDGE_ALPHAS[0]) or np.isclose(a, RIDGE_ALPHAS[-1])


# Gradient-boosted trees: 300 small trees (at most 15 leaves, at least 30 papers per leaf) with a
# slow learning rate and L2 regularisation, settings that limit overfitting on ~1,400 training
# papers. They handle missing values directly.
def trees():
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.04, max_leaf_nodes=15,
                                         min_samples_leaf=30, l2_regularization=1.0, random_state=0)


# Fit on the training papers using only the columns `cols`, predict the test papers, and return
# Spearman rho (rank agreement with the actual percentile) and the AUC for picking the top 10%.
def score(model, cols):
    model.fit(X.loc[train, cols], y[train])
    p = model.predict(X.loc[test, cols])
    return spearmanr(p, y[test]).statistic, roc_auc_score(D.loc[test, "top10"], p)


# Both models with all features.
lin = ridge()
rr, ra = score(lin, FEATURES)
tr_, ta = score(trees(), FEATURES)
print(f"ridge: test rho {rr:.3f}, AUC {ra:.3f} (penalty {lin[-1].alpha_:.1f}"
      + (", AT THE GRID EDGE -> widen RIDGE_ALPHAS)" if alpha_at_edge(lin[-1].alpha_) else ", inside the grid)"))
print(f"trees: test rho {tr_:.3f}, AUC {ta:.3f}")


---
## 7 · Which features go with high citations?

Ridge regression on all modelled papers (no split), features standardised: each coefficient is the change in citation percentile for a one-standard-deviation change in the feature **holding the other features fixed**. For a yes/no feature with share p, one standard deviation is √(p(1−p)), so the difference between papers with and without it is the coefficient divided by √(p(1−p)): larger than the coefficient, the more so the rarer the feature. 95% intervals from 300 bootstrap refits. The table lists them by size of effect.

**Journal coding.** Every refereed paper has exactly one journal flag, so "refereed" is the sum of the six journal flags and the separate journal effects cannot all be estimated. Here the Astrophysical Journal itself is the baseline: `refereed` and `journal_apj` (ApJ + ApJS) are replaced by a `preprint` flag (not yet refereed) and a separate `journal_apjs` flag, since ApJS mostly publishes catalogues and surveys. Each journal effect then reads "this journal vs. ApJ, everything else equal". The association model therefore has as many features as the prediction models, with the journal block recoded.

In [ ]:
from common.trending_shared import INK, SECONDARY, MUTED, SURFACE, GRID, _wrapped_text

# Journal coding for the associations. Every refereed paper has exactly one journal flag, so
# "refereed" equals the sum of the six journal flags: with all seven in the model the separate journal
# effects cannot be told apart, and ridge splits them in a way that depends on the penalty. Here the
# Astrophysical Journal itself is the baseline: "refereed" and journal_apj (ApJ + ApJS) are replaced by
# a "preprint" flag (not yet refereed) and a separate ApJ Supplement flag, since ApJS mostly publishes
# catalogues and surveys. A journal's effect then reads "this journal vs. ApJ, everything else equal".
# The prediction models (sections 6, 8, 9) keep the original coding; for them it makes no difference.
APJS = S["bibcode"].map(lambda b: "Astrophysical Journal Supplement" in ((meta.get(b) or {}).get("pub") or ""))


def assoc_coding(df, apjs):
    """`df` with the journal block recoded against ApJ; `apjs`: the ApJS flag for the same rows."""
    return df.drop(columns=["refereed", "journal_apj"]).assign(preprint=1 - df["refereed"],
                                                               journal_apjs=np.asarray(apjs, float))


XA = assoc_coding(X, APJS[M].to_numpy())   # X holds the modelled papers (M) in order
ASSOC = list(XA.columns)          # the features of the association model

# Standardise the features (missing values set to the median), so each coefficient is per standard
# deviation of its feature and the coefficients can be compared.
imp = SimpleImputer(strategy="median")
Z = imp.fit_transform(XA)
Z = (Z - Z.mean(0)) / np.where(Z.std(0) > 0, Z.std(0), 1)
# Penalty strength chosen by cross-validation on all papers, then kept fixed for the point estimate
# and for every bootstrap refit.
alpha = ridge().fit(XA, y)[-1].alpha_
print(f"Ridge penalty for the associations: alpha = {alpha:.1f}"
      + ("  AT THE GRID EDGE -> widen RIDGE_ALPHAS" if alpha_at_edge(alpha) else "  (inside the grid)"))
from sklearn.linear_model import Ridge
coef = Ridge(alpha=alpha).fit(Z, y).coef_
rng = np.random.default_rng(0)
# 300 refits on resampled papers (drawn with replacement); the 2.5th and 97.5th percentiles of each
# coefficient give its 95% interval.
boot = np.array([Ridge(alpha=alpha).fit(Z[i], y[i]).coef_
                 for i in (rng.integers(0, len(y), len(y)) for _ in range(300))])
group_of = {c: g for g, cs in GROUPS.items() for c in cs} | {"preprint": "Form", "journal_apjs": "Form"}
df_assoc = pd.DataFrame({"feature": ASSOC, "group": [group_of[c] for c in ASSOC], "effect": coef,
                         "low": np.percentile(boot, 2.5, 0), "high": np.percentile(boot, 97.5, 0),
                         "coverage %": XA.notna().mean().mul(100).round(0).to_numpy()})
df_assoc["CI excludes 0"] = (df_assoc["low"] > 0) | (df_assoc["high"] < 0)
# Sort by size of effect, whatever its sign.
df_assoc = df_assoc.reindex(df_assoc["effect"].abs().sort_values(ascending=False).index).reset_index(drop=True)
display(df_assoc.round(2).head(30))
df_assoc.round(3).to_csv(f"{PLOTS_DIR}/associations.csv", index=False)

In [ ]:
# Readable feature names, and the house font of the figures below (common/figstyle.py).
from common.figstyle import STYLE_RC as FIG_RC
plt.rcParams["font.family"] = FIG_RC["font.family"]   # shared house font (common/figstyle.py)
# Readable names for the features.
LABEL = {"topic_rank_earliest": "Later on its topic (rank)", "log_topic_papers_before": "Papers on its topic already out",
         "n_programmes": "JWST programmes used", "newest_cycle": "Newest cycle of data used",
         "data_age_years": "Age of the data", "public_survey_data": "Uses public ERO/ERS/treasury data",
         "gto_data": "Uses GTO data", "log_n_authors": "Number of authors (log)", "large_collab": "50+ authors",
         "n_countries": "Countries in the team", "n_regions": "Continents in the team", "fa_europe": "First author in Europe",
         "fa_n_america": "First author in N. America", "fa_asia": "First author in Asia",
         "fa_log_prior_papers": "First author's earlier papers (log)", "fa_career_years": "First author's career length",
         "fa_shared_name": "First author's name shared (count is a lower limit)",
         "catalogue_or_release": "Catalogue / data release / survey paper",
         "instrument_paper": "Instrument or calibration paper", "abs_claims": "Claims of firsts, surprises (abstract)",
         "abs_very_high_z": "Very high redshift (abstract)", "abs_hedging": "Hedging words (abstract)",
         "abs_log_words": "Abstract length", "abs_numbers_per_100w": "Numbers in the abstract",
         "refereed": "Refereed", "preprint": "Preprint, not yet refereed (vs. ApJ)",
         "journal_nature_science": "Nature / Science family (vs. ApJ)", "journal_apjl": "ApJ Letters (vs. ApJ)",
         "journal_apj": "ApJ / ApJS", "journal_apjs": "ApJ Supplement (vs. ApJ)", "journal_aa": "A&A (vs. ApJ)", "journal_mnras": "MNRAS (vs. ApJ)",
         "journal_other": "Other journal (vs. ApJ)",
         "letter": "Letter format", "log_pages": "Pages (log)", "log_references": "References (log)",
         "open_data_code": "Releases data or code", "title_words": "Title length", "title_separator": "Title with separator",
         "title_question": "Title is a question", "title_survey_name": "Survey name in title",
         "title_catchy": "Catchy title",
         "abs_nircam": "NIRCam in the abstract", "abs_nirspec": "NIRSpec in the abstract",
         "abs_miri": "MIRI in the abstract", "abs_niriss": "NIRISS in the abstract", "abs_prism": "Prism in the abstract",
         "abs_ifu": "IFU in the abstract", "abs_grism": "Grism in the abstract", "abs_alma": "ALMA in the abstract",
         "abs_hst": "HST in the abstract"}
label = lambda f: LABEL.get(f, f.replace("topic_", "Topic: ").replace("abs_", "Abstract: ").replace("_", " "))


---
## 8 · How well do the predictions work?

The gradient-boosted trees with all features, checked three ways:

- **A · Predicted vs. actual percentile.** Papers in ten equal bins of predicted percentile, against their mean actual percentile (95% interval). Training papers are predicted by the model fitted on them (in-sample); test papers (posted from `TEST_FROM` on) were never seen. A perfect model follows the diagonal; an uninformative one is a flat line at 50.
- **B · Finding the top 10%.** Rank papers by prediction and read off what share of the true top-10% papers are among the highest-ranked x% (a gains curve; the diagonal is random picking).
- **C · Cross-validation.** *Forward in time:* for each quarter from 2024 on, train on everything posted before it and predict that quarter, the way the model would be used. *Across months:* 5-fold cross-validation holding out whole posting months (band: mean ± 1 s.d. of the folds).

In [ ]:

from sklearn.model_selection import GroupKFold
BLUE, BLUE_LIGHT, BLUE_MID, ORANGE = "#2a78d6", "#9ec5f4", "#6da7ec", "#eb6834"

# in-sample (training) and out-of-sample (test) predictions of the time-split model
# The time-split model: predictions for the papers it was trained on (in-sample, optimistic) and for
# the later test papers (out-of-sample, the honest score).
mdl = trees().fit(X[train], y[train])
p_train, p_test = mdl.predict(X[train]), mdl.predict(X[test])

# 5-fold cross-validation holding out whole posting months: out-of-fold predictions for every paper
# 5-fold cross-validation with whole posting months held out (GroupKFold on the cohort): every paper
# gets a prediction from a model that never saw its month.
oof, fold_scores = np.full(len(y), np.nan), []
for k, (a, b) in enumerate(GroupKFold(n_splits=5).split(X, y, groups=D["cohort"])):
    oof[b] = trees().fit(X.iloc[a], y[a]).predict(X.iloc[b])
    fold_scores.append((spearmanr(oof[b], y[b]).statistic, roc_auc_score(D["top10"].iloc[b], oof[b])))
fold_scores = np.array(fold_scores)

# forward in time: train on everything before each quarter, predict that quarter
# Forward in time: for every quarter from 2024 on (with at least 40 papers), train on all earlier
# papers and predict that quarter. The 95% interval of rho comes from 500 bootstrap resamples of the
# quarter's papers.
quarter = pd.PeriodIndex(D["month"], freq="Q")
rng = np.random.default_rng(0)
fwd = []
for q in sorted(set(quarter[quarter >= pd.Period("2024Q1")])):
    tr, te = (quarter < q), (quarter == q)
    if te.sum() < 40:
        continue
    pq = trees().fit(X[tr], y[tr]).predict(X[te])
    yq = y[te]
    bs = [spearmanr(pq[i], yq[i]).statistic for i in (rng.integers(0, te.sum(), te.sum()) for _ in range(500))]
    fwd.append({"quarter": str(q), "n": int(te.sum()), "rho": spearmanr(pq, yq).statistic,
                "low": np.percentile(bs, 2.5), "high": np.percentile(bs, 97.5),
                "auc": roc_auc_score(D["top10"][te], pq)})
df_fwd = pd.DataFrame(fwd)

# One table with the four ways of scoring the model.
summary = pd.DataFrame({
    "Spearman ρ": [spearmanr(p_train, y[train]).statistic, spearmanr(p_test, y[test]).statistic,
                   fold_scores[:, 0].mean(), df_fwd["rho"].mean()],
    "AUC top 10%": [roc_auc_score(D["top10"][train], p_train), roc_auc_score(D["top10"][test], p_test),
                    fold_scores[:, 1].mean(), df_fwd["auc"].mean()]},
    index=["training (in-sample)", "test (time split)", "5-fold CV by month (mean)", "forward by quarter (mean)"])
display(summary.round(3))
display(df_fwd.round(3))


In [ ]:

def binned(pred, true, n=10):
    """Mean predicted and mean actual percentile in n equal-count bins of the prediction (+ 95% CI)."""
    order = np.argsort(pred)
    out = []
    for idx in np.array_split(order, n):
        t = true[idx]
        out.append((pred[idx].mean(), t.mean(), 1.96 * t.std(ddof=1) / np.sqrt(len(t))))
    return np.array(out)


def gains(pred, is_top):
    """Share of the true top papers found among the highest-ranked x% of predictions."""
    order = np.argsort(-pred)
    found = np.cumsum(np.asarray(is_top)[order]) / np.sum(is_top)
    return np.r_[0, np.arange(1, len(pred) + 1) / len(pred)] * 100, np.r_[0, found] * 100


# Shared axis style: light gridlines, no top and right spines, grey tick labels.
def style(ax):
    ax.tick_params(axis="both", length=0, labelsize=8.5, colors=SECONDARY)
    ax.grid(color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)


# Three panels; the third is wider for its ten quarters.
fig, axes = plt.subplots(1, 3, figsize=(13.5, 5.2), facecolor=SURFACE,
                         gridspec_kw={"width_ratios": [1, 1, 1.25], "wspace": 0.28})
axA, axB, axC = axes

# A · calibration
# Panel A, calibration: the dashed diagonal is a perfect model, the horizontal line at 50 a model
# with no skill. Points: mean actual percentile in ten equal-size bins of the prediction, with 95%
# intervals.
axA.plot([0, 100], [0, 100], color=MUTED, lw=1, ls=(0, (3, 2)), zorder=1)
axA.axhline(50, color=GRID, lw=1, zorder=1)
for pred, true, col, lab in ((p_train, y[train], BLUE_LIGHT, f"training, in-sample ({train.sum():,})"),
                             (p_test, y[test], BLUE, f"test, unseen ({test.sum():,})")):
    b = binned(pred, true)
    axA.errorbar(b[:, 0], b[:, 1], yerr=b[:, 2], fmt="o-", color=col, ms=6, lw=2, capsize=0,
                 mec=SURFACE, mew=1.2, zorder=3, label=lab)
axA.set_xlim(0, 100), axA.set_ylim(0, 100)
axA.set_xlabel("Predicted percentile (bin mean)", fontsize=9, color=SECONDARY)
axA.set_ylabel("Actual percentile (bin mean)", fontsize=9, color=SECONDARY)
axA.text(98, 89, "perfect", ha="right", va="top", fontsize=8, color=MUTED)
axA.text(98, 51.5, "no skill", ha="right", va="bottom", fontsize=8, color=MUTED)
axA.legend(loc="upper left", fontsize=8, frameon=False)
style(axA)

# B · gains for the top 10%
# Panel B, gains curve: going down the list ranked by the model, the share of true top-10% papers
# found. The diagonal is random picking. The dotted guide reads off the top 20% of the test list.
axB.plot([0, 100], [0, 100], color=MUTED, lw=1, ls=(0, (3, 2)), zorder=1)
for pred, top_, col, lab in ((p_train, D["top10"][train], BLUE_LIGHT, "training, in-sample"),
                             (oof, D["top10"], BLUE_MID, "5-fold CV by month"),
                             (p_test, D["top10"][test], BLUE, "test, unseen")):
    gx, gy = gains(pred, top_)
    axB.plot(gx, gy, color=col, lw=2, zorder=3, label=f"{lab} (AUC {roc_auc_score(top_, pred):.2f})")
gx, gy = gains(p_test, D["top10"][test])
k = np.searchsorted(gx, 20)
axB.plot([20, 20, 0], [0, gy[k], gy[k]], color=MUTED, lw=0.8, ls=":", zorder=2)
axB.text(21.5, gy[k] - 1, f"top 20% of predictions hold\n{gy[k]:.0f}% of the test top-10% papers",
         fontsize=8, color=SECONDARY, va="top")
axB.text(84, 76, "random", fontsize=8, color=MUTED, ha="left", va="top")
axB.set_xlim(0, 100), axB.set_ylim(0, 102)
axB.set_xlabel("Papers ranked highest by the model (%)", fontsize=9, color=SECONDARY)
axB.set_ylabel("True top-10% papers found (%)", fontsize=9, color=SECONDARY)
axB.legend(loc="lower right", fontsize=8, frameon=False)
style(axB)

# C · cross-validation over time
# Panel C, stability: rho per quarter predicted from earlier papers only (dots, 95% bars) against
# the 5-fold cross-validation mean and spread (band).
xs = np.arange(len(df_fwd))
mu, sd = fold_scores[:, 0].mean(), fold_scores[:, 0].std()
band = axC.axhspan(mu - sd, mu + sd, color=BLUE_LIGHT, alpha=0.35, lw=0, zorder=1)
axC.axhline(mu, color=BLUE_MID, lw=1.2, zorder=1)
axC.axhline(0, color=GRID, lw=1)
axC.vlines(xs, df_fwd["low"], df_fwd["high"], color=BLUE_LIGHT, lw=4, zorder=2)
dots, = axC.plot(xs, df_fwd["rho"], "o", color=BLUE, ms=8, mec=SURFACE, mew=1.5, zorder=3)
axC.legend([dots, (band, plt.Line2D([], [], color=BLUE_MID, lw=1.2))],
           ["each quarter, trained on earlier papers", f"5-fold CV by month: ρ = {mu:.2f} ± {sd:.2f} (mean ± s.d.)"],
           loc="lower left", fontsize=8, frameon=False)
for x_, r in zip(xs, df_fwd.itertuples()):
    axC.text(x_, r.high + 0.015, f"{r.n}", ha="center", va="bottom", fontsize=7, color=MUTED)
axC.set_xticks(xs, [f"Q{q[-1]}\n{q[:4]}" if q.endswith("1") or i == 0 else f"Q{q[-1]}"
                    for i, q in enumerate(df_fwd["quarter"])], fontsize=8)
axC.set_ylim(-0.05, 0.9)
axC.text(len(xs) - 0.5, 0.875, "papers per quarter above each bar", ha="right", va="top", fontsize=7, color=MUTED)
axC.set_xlim(-0.6, len(xs) - 0.4)
axC.set_ylabel("Spearman ρ, predicted vs. actual", fontsize=9, color=SECONDARY)
style(axC)
axC.grid(axis="x", visible=False)

# Panel headings, figure title and footnote; saved as PNG (400 dpi) and PDF.
heads = [("A · Predictions follow the actual ranking", "Mean actual vs. predicted percentile, 10 bins"),
         ("B · Most top papers are flagged early", "Gains curve for the top 10% of each month"),
         ("C · Stable when predicting ahead in time", "Each quarter predicted by a model trained on earlier papers (95% CI)")]
for ax, (h, sub) in zip(axes, heads):
    ax.text(0, 1.13, h, transform=ax.transAxes, fontsize=11, color=INK, va="bottom")
    ax.text(0, 1.06, sub, transform=ax.transAxes, fontsize=8.3, color=SECONDARY, va="bottom")
fig.subplots_adjust(left=0.05, right=0.99, top=0.78, bottom=0.21)
fig.text(0.05, 0.955, "How well can a paper's features predict its citations?", fontsize=14, color=INK, va="bottom")
_wrapped_text(fig, f"{M.sum():,} JWST galaxy articles and preprints that use JWST data (in ADS's JWST bibliography or "
              f"explicit data use in the text) posted before {cut}; "
              f"target: citation percentile among papers posted the same month (citations as of {CITES_AS_OF}). "
              f"Gradient-boosted trees on {len(FEATURES)} features (timing, data, team, content, form). Training: cohorts before "
              f"{TEST_FROM} ({train.sum():,} papers); test: {TEST_FROM} on ({test.sum():,}). Test: ρ = "
              f"{summary.loc['test (time split)', 'Spearman ρ']:.2f}, AUC = {summary.loc['test (time split)', 'AUC top 10%']:.2f}; "
              f"in-sample fit: ρ = {summary.loc['training (in-sample)', 'Spearman ρ']:.2f}. 5-fold CV holds out whole posting months. "
              "AUC: chance of ranking a top-10% paper above another (0.5 = chance).",
              0.05, 0.99, 0.01, fontsize=7.5, color=MUTED, linespacing=1.5)
fig.savefig(f"{PLOTS_DIR}/top_cited_prediction_check.png", dpi=400, facecolor=SURFACE)
fig.savefig(f"{PLOTS_DIR}/top_cited_prediction_check.pdf", facecolor=SURFACE)
plt.show()


---
## 9 · Robustness tests

How much do the conclusions of sections 6-8 depend on choices made along the way? Each test changes one thing, refits, and scores on the same unseen test papers (posted from `TEST_FROM` on) unless it says otherwise. Every result is collected in one table at the end of the section.

| Test | Question it answers |
|---|---|
| 1 · Tree settings | Do the scores depend on how the trees are built and pruned (size, minimum papers per leaf, learning rate, L2 penalty)? |
| 2 · Number of trees, nested selection | Do we overfit by adding too many trees? What score do we get if the settings are chosen without ever looking at the test period? |
| 3 · Chance | How much would the scores move with a different set of test papers, or of training papers? |
| 4 · Tree family | Do random forests (a different kind of tree model) give the same answer? |
| 5 · Null test | With the citation ranks shuffled, does the skill vanish, as it must if nothing leaks? |
| 6 · Information at posting time | How much skill is left using only what is known when a paper first appears on arXiv? |
| 7 · Data quality | Does the model work on refereed papers only (complete features), on preprints, and when the first author's history is complete (name not shared in ADS)? |
| 8 · Target and design | Do other cohort sizes, minimum ages, time splits and targets change the picture? |
| 9 · Linear associations | Are the effects of section 7 stable against the penalty strength, a sparse model, resampling, and correlated features? |
| 10 · Self-citations | Do the results survive when citations from the paper's own authors are removed? |

In [ ]:
# ── 10 · Robustness tests: shared helpers ───────────────────────────────────────────────────
import itertools, warnings, unicodedata
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, ElasticNetCV
warnings.filterwarnings("ignore")   # convergence and future-version warnings would only clutter the outputs

# The tree settings used in sections 6-8 ("reference"); every test below varies these.
BASE = dict(leaves=15, min_leaf=30, lr=0.04, n_trees=300, l2=1.0)


def hgb(leaves=15, min_leaf=30, lr=0.04, n_trees=300, l2=1.0, seed=0):
    """Gradient-boosted trees with explicit settings; hgb() is identical to trees() of section 6.
    leaves   = max_leaf_nodes: the largest number of end nodes ("leaves") one tree may have, i.e. its size
    min_leaf = min_samples_leaf: the fewest training papers allowed in a leaf -- the main pruning knob:
               a large value forbids splits that isolate small groups of papers
    lr       = learning_rate: how strongly each new tree corrects the errors of the previous ones
    n_trees  = max_iter: how many trees are added one after the other
    l2       = l2_regularization: pulls the value stored in each leaf towards zero"""
    return HistGradientBoostingRegressor(max_iter=n_trees, learning_rate=lr, max_leaf_nodes=leaves,
                                         min_samples_leaf=min_leaf, l2_regularization=l2, random_state=seed)


def evaluate(model, cols=None, Xd=None, yd=None, tr=None, te=None, top=None):
    """Fit `model` on the training papers, predict the test papers, and return
    (Spearman rho between predicted and actual target, AUC for picking the top-10% papers).
    Every argument defaults to section 6: features X, target y, the time split, the top-10% flag."""
    Xd = X if Xd is None else Xd
    yd = y if yd is None else yd
    tr = train if tr is None else tr
    te = test if te is None else te
    top = D["top10"].to_numpy() if top is None else top
    cols = FEATURES if cols is None else cols
    # Learn only from the training papers (and only from the chosen columns) ...
    model.fit(Xd.loc[tr, cols], yd[tr])
    # ... then predict papers the model has never seen.
    p = model.predict(Xd.loc[te, cols])
    return spearmanr(p, yd[te]).statistic, roc_auc_score(top[te], p)


# One row per test variant; `low`/`high` hold a range when the variant has one (a grid, resamples).
# The summary table and figure at the end are built from this list.
ROBUST = []


def record(test, variant, rho, auc, low=None, high=None, papers=None):
    ROBUST.append({"test": test, "variant": variant, "rho": rho, "AUC": auc, "low": low, "high": high,
                   "papers": papers})


# The reference score every test is compared with (it must equal section 6's 'all features' trees
# score).
BASE_RHO, BASE_AUC = evaluate(hgb())
print(f"Reference (settings of sections 6-8): test rho = {BASE_RHO:.3f}, AUC = {BASE_AUC:.3f} "
      f"({train.sum():,} training, {test.sum():,} test papers)")

### Test 1 · Tree settings (pruning and construction)

A grid of 225 settings: 5 tree sizes × 5 minimum leaf sizes × 3 learning rates × 3 L2 penalties. The number of trees is set so that learning rate × trees stays about 12 (a smaller step needs more trees to get as far). The table shows the median score for each value of each parameter, with all other parameters varying: a parameter whose medians barely change does not matter.

In [ ]:
# ── Test 1 · grid over tree construction and pruning ──
# (named TREE_GRID: GRID is already the gridline colour imported from trending_shared)
TREE_GRID = {"leaves": [4, 8, 15, 31, 63], "min_leaf": [5, 15, 30, 60, 120], "lr": [0.02, 0.04, 0.1], "l2": [0.0, 1.0, 10.0]}
rows_grid = []
# itertools.product walks through every combination of the four lists: 5 x 5 x 3 x 3 = 225 fits.
for leaves, min_leaf, lr, l2 in itertools.product(*TREE_GRID.values()):
    n_trees = int(round(12 / lr))                      # 600 trees at lr 0.02, 300 at 0.04, 120 at 0.1
    rho, auc = evaluate(hgb(leaves, min_leaf, lr, n_trees, l2))
    rows_grid.append({"leaves": leaves, "min_leaf": min_leaf, "lr": lr, "l2": l2, "n_trees": n_trees,
                      "rho": rho, "AUC": auc})
# One row per setting; saved so the whole grid can be inspected or plotted later.
df_grid = pd.DataFrame(rows_grid)
df_grid.round(4).to_csv(f"{PLOTS_DIR}/robustness_tree_grid.csv", index=False)

print(f"{len(df_grid)} settings: test rho {df_grid['rho'].min():.3f} to {df_grid['rho'].max():.3f} "
      f"(median {df_grid['rho'].median():.3f}); AUC {df_grid['AUC'].min():.3f} to {df_grid['AUC'].max():.3f}")
# median score per value of each parameter (the other three varying): which knob matters?
display(pd.concat({p: df_grid.groupby(p)[["rho", "AUC"]].median() for p in TREE_GRID}).round(3))
print("The five worst settings:")
display(df_grid.nsmallest(5, "rho").round(3))

# for the summary: the whole grid (median and 5-95% range), and the settings that matter most
# Median and 5th-95th percentile range of a set of scores.
q = lambda s: (s.median(), np.percentile(s, 5), np.percentile(s, 95))
m_, lo_, hi_ = q(df_grid["rho"])
record("1 Tree settings", "all 225 settings (median, 5-95% range)", m_, df_grid["AUC"].median(), lo_, hi_, test.sum())
for v in (5, 30, 60, 120):
    g = df_grid[df_grid["min_leaf"] == v]
    m_, lo_, hi_ = q(g["rho"])
    record("1 Tree settings", f"min {v} papers per leaf", m_, g["AUC"].median(), lo_, hi_, test.sum())
for v in (4, 63):
    g = df_grid[df_grid["leaves"] == v]
    m_, lo_, hi_ = q(g["rho"])
    record("1 Tree settings", f"max {v} leaves per tree", m_, g["AUC"].median(), lo_, hi_, test.sum())

### Test 2 · Number of trees, and settings chosen without the test period

Boosting adds trees one after another; too many start fitting noise in the training papers (overfitting), which shows as a slow decline on the test papers. Then a *nested* selection: the training period is itself split in time (its last 9 months act as a validation set), the best of 64 settings is picked there, the model is refitted on the whole training period and scored once on the test period. That is the honest score of a fully tuned model, since the test papers played no part in choosing it.

In [ ]:
# ── Test 2a · learning curve in the number of trees (other settings as the reference) ──
rows_curve = []
for n in [25, 50, 100, 200, 300, 600, 1200, 2400]:
    rho, auc = evaluate(hgb(n_trees=n))
    rows_curve.append({"n_trees": n, "rho": rho, "AUC": auc})
df_curve = pd.DataFrame(rows_curve)
display(df_curve.round(3))
for r in rows_curve:
    if r["n_trees"] in (50, 100, 1200, 2400):
        record("2 Number of trees", f"{r['n_trees']} trees", r["rho"], r["AUC"], papers=test.sum())

# ── Test 2b · nested selection: choose the settings on validation months inside the training period ──
VAL_FROM = (pd.Period(TEST_FROM, "M") - 9).strftime("%Y-%m")       # validation = the last 9 months before the test period
inner_tr = train & (D["cohort"] < VAL_FROM).to_numpy()             # fit candidates on the earlier training papers
inner_va = train & (D["cohort"] >= VAL_FROM).to_numpy()            # ...and compare them on the later ones
cands = []
for leaves, min_leaf, n in itertools.product([4, 8, 15, 31], [5, 15, 30, 60], [50, 100, 200, 400]):
    rho_va, _ = evaluate(hgb(leaves, min_leaf, n_trees=n), tr=inner_tr, te=inner_va)
    cands.append((rho_va, leaves, min_leaf, n))
# Tuples compare element by element, so max() picks the highest validation rho.
best = max(cands)                                                   # highest validation rho
rho, auc = evaluate(hgb(best[1], best[2], n_trees=best[3]))          # refit on all training papers, score on test
print(f"Nested selection: validation months {VAL_FROM} to {TEST_FROM} (excl.), {inner_va.sum()} papers.")
print(f"  chosen: {best[1]} leaves, min {best[2]} papers/leaf, {best[3]} trees (validation rho {best[0]:.3f})")
print(f"  test: rho {rho:.3f}, AUC {auc:.3f}   (reference {BASE_RHO:.3f}, {BASE_AUC:.3f})")
record("2 Number of trees", "settings chosen without the test period", rho, auc, papers=test.sum())

### Test 3 · How much do the scores move by chance?

(a) The test papers are one sample of possible papers: resampling them with replacement (1,000 times) and rescoring the same model gives a 95% interval on the test scores. (b) The training papers too: refitting on 20 resamples of the training set shows how much the model itself would change.

In [ ]:
# ── Test 3a · uncertainty from the test papers: resample them, keep the fitted model ──
m0 = hgb().fit(X[train], y[train])
p0, yt, tt = m0.predict(X[test]), y[test], D["top10"].to_numpy()[test]
rng = np.random.default_rng(1)
bs = []
for _ in range(1000):
    i = rng.integers(0, len(yt), len(yt))                 # one resample of the test papers
    bs.append((spearmanr(p0[i], yt[i]).statistic, roc_auc_score(tt[i], p0[i])))
bs = np.array(bs)
# The middle 95% of the resampled scores = the 95% interval.
RHO_CI = np.percentile(bs[:, 0], [2.5, 97.5])            # used as the grey band of the summary figure
AUC_CI = np.percentile(bs[:, 1], [2.5, 97.5])
print(f"(a) test papers resampled: rho {BASE_RHO:.3f} [{RHO_CI[0]:.3f}, {RHO_CI[1]:.3f}], "
      f"AUC {BASE_AUC:.3f} [{AUC_CI[0]:.3f}, {AUC_CI[1]:.3f}] (95%)")
record("3 Chance", "test papers resampled (95% interval)", BASE_RHO, BASE_AUC, *RHO_CI, test.sum())

# ── Test 3b · uncertainty from the training papers: refit on 20 bootstrap resamples ──
# Row numbers of the training papers (reused by tests 5 and 6).
tr_idx = np.where(train)[0]
refits = []
for s in range(20):
    b = np.random.default_rng(s).choice(tr_idx, len(tr_idx), replace=True)   # resampled training set
    p = hgb(seed=s).fit(X.iloc[b], y[b]).predict(X[test])
    refits.append((spearmanr(p, yt).statistic, roc_auc_score(tt, p)))
refits = np.array(refits)
print(f"(b) training papers resampled (20 refits): rho {refits[:, 0].mean():.3f} ± {refits[:, 0].std():.3f}, "
      f"AUC {refits[:, 1].mean():.3f} ± {refits[:, 1].std():.3f}")
print("    (a resample repeats some papers and misses about a third, so it trains on less information: "
      "its mean is slightly lower than the reference)")
record("3 Chance", "training papers resampled (mean ± 2 s.d.)", refits[:, 0].mean(), refits[:, 1].mean(),
       refits[:, 0].mean() - 2 * refits[:, 0].std(), refits[:, 0].mean() + 2 * refits[:, 0].std(), test.sum())

### Test 4 · A different tree family: random forests

Random forests grow many deep trees, each on a random subset of papers and features, and average them, instead of adding small corrective trees one after another. They need complete data, so missing values are replaced by the training-period median.

In [ ]:
# ── Test 4 · random forests with three leaf sizes ──
Xf = X.fillna(X[train].median())          # median of the TRAINING papers only, so nothing from the test period leaks in
for leaf in (5, 20, 50):
    # 500 trees; each split considers a random third of the features (max_features=0.33), which
    # decorrelates the trees; n_jobs=-1 uses all processor cores.
    rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=leaf, max_features=0.33, n_jobs=-1, random_state=0)
    rho, auc = evaluate(rf, Xd=Xf)
    print(f"random forest, min {leaf} papers per leaf: rho {rho:.3f}, AUC {auc:.3f}")
    record("4 Tree family", f"random forest, min {leaf} papers per leaf", rho, auc, papers=test.sum())

### Test 5 · Null test

The training papers' percentiles are shuffled *within each posting month*, so every month keeps the same values but they land on the wrong papers. A model fitted to that can learn nothing real, so its test ρ should scatter around zero. A clearly positive value would mean information leaks from the target into the features.

In [ ]:
# ── Test 5 · null test: shuffled targets, 10 repeats ──
null = []
groups_tr = D.iloc[tr_idx].groupby("cohort").indices          # positions (within the training rows) of each month
for k in range(10):
    yp = y.copy()
    rng = np.random.default_rng(k)
    for cohort, pos in groups_tr.items():
        sel = tr_idx[pos]                                       # rows of this month in the full table
        yp[sel] = rng.permutation(yp[sel])                      # shuffle the percentiles among them
    null.append(evaluate(hgb(), yd=yp))                         # test targets stay the real ones
# Rows = repeats; columns = (rho, AUC).
null = np.array(null)
print(f"shuffled target (10 repeats): rho {null[:, 0].mean():+.3f} ± {null[:, 0].std():.3f}, "
      f"AUC {null[:, 1].mean():.3f} ± {null[:, 1].std():.3f}  (no skill: rho 0, AUC 0.5)")
record("5 Null test", "citation ranks shuffled (mean ± 2 s.d.)", null[:, 0].mean(), null[:, 1].mean(),
       null[:, 0].mean() - 2 * null[:, 0].std(), null[:, 0].mean() + 2 * null[:, 0].std(), test.sum())

### Test 6 · Only what is known when the paper is posted

Journal, refereed status, Letter format, page count and reference count are only certain once a paper is published, often months after it appears on arXiv, and a journal's decision may itself anticipate impact. Variant (a) drops them. Variant (b) also removes the information carried by *missing* values (affiliations are missing for preprints, so "missing" quietly means "not yet published"): every missing value is filled with the training median, so the model cannot tell missing from typical.

In [ ]:
# ── Test 6 · features available at arXiv posting only ──
POST_PUB = [c for c in FEATURES if c.startswith("journal_")] + ["refereed", "letter", "log_pages", "log_references"]
# Everything else is known when the paper first appears on arXiv: topic, abstract wording, data,
# team, title.
AT_POSTING = [c for c in FEATURES if c not in POST_PUB]
rho, auc = evaluate(hgb(), cols=AT_POSTING)
print(f"(a) without the {len(POST_PUB)} publication features: rho {rho:.3f}, AUC {auc:.3f}")
record("6 Posting time", "without journal, refereed, pages, references", rho, auc, papers=test.sum())

Xm = X.fillna(X[train].median())                                # missingness no longer visible to the model
rho, auc = evaluate(hgb(), cols=AT_POSTING, Xd=Xm)
print(f"(b) ... and missing values filled with the training median: rho {rho:.3f}, AUC {auc:.3f}")
record("6 Posting time", "same, and missing values filled", rho, auc, papers=test.sum())

### Test 7 · Data quality

(a) Refereed papers only, where every feature is available (train and test restricted to them). (b) The reference model scored on the test-period *preprints* only, the papers with the least information. (c) Papers whose first author's name is not shared with anyone else in ADS, so the author-history counts are complete rather than lower limits (section 3). (d) The reference model without the three author-history features.

In [ ]:
# ── Test 7 · subsets with complete or reliable data ──
# True for refereed (journal-published) papers, which have complete metadata in ADS.
refd = (X["refereed"] == 1).to_numpy()
rho, auc = evaluate(hgb(), tr=train & refd, te=test & refd)
print(f"(a) refereed papers only ({(train & refd).sum()} train, {(test & refd).sum()} test): rho {rho:.3f}, AUC {auc:.3f}")
record("7 Data quality", "refereed papers only", rho, auc, papers=int((test & refd).sum()))

rho, auc = evaluate(hgb(), te=test & ~refd)
print(f"(b) reference model, test preprints only ({(test & ~refd).sum()}): rho {rho:.3f}, AUC {auc:.3f}")
record("7 Data quality", "test-period preprints only", rho, auc, papers=int((test & ~refd).sum()))

# first author's name not shared in ADS: complete history counts (rule 'single' in section 3)
single_fa = D["bibcode"].map(FA.set_index("bibcode")["rule"]).eq("single").to_numpy()
rho, auc = evaluate(hgb(), tr=train & single_fa, te=test & single_fa)
print(f"(c) first author's name not shared ({single_fa.mean():.0%} of papers): rho {rho:.3f}, AUC {auc:.3f}")
record("7 Data quality", "first author's name not shared", rho, auc, papers=int((test & single_fa).sum()))

# All features except the three that come from the first author's history.
no_hist = [c for c in FEATURES if c not in ("fa_log_prior_papers", "fa_career_years", "fa_shared_name")]
rho, auc = evaluate(hgb(), cols=no_hist)
print(f"(d) without the first author's history: rho {rho:.3f}, AUC {auc:.3f}")
record("7 Data quality", "without first-author history", rho, auc, papers=test.sum())

### Test 8 · Target and design choices

The target is rebuilt with other choices, then the reference model is refitted: smaller or larger cohorts (pooling months below 15 or 50 papers instead of 25), quarterly instead of monthly cohorts, a minimum age of 6 or 12 months instead of 3, the time split moved 6 months earlier or 3 months later, and a different target altogether (log citations minus the cohort mean, instead of the rank). The "top" flag is also moved to the top 5% and the top 20%.

In [ ]:
# ── Test 8 · rebuild the target with other choices ──
def build_target(min_cohort=MIN_COHORT, min_age=MIN_AGE_MONTHS, quarterly=False, kind="pct", cites=None):
    """Like section 5, with other choices. Returns (papers kept [over S], target [over the kept papers],
    cohort label = the cohort's first month [over the kept papers])."""
    cites = S["citations"] if cites is None else cites
    cut_ = (pd.Period(CITES_AS_OF, "M") - min_age).strftime("%Y-%m")
    ok = S["month"].notna() & (S["month"] < cut_) & cites.notna()
    unit = S.loc[ok, "month"]
    if quarterly:   # label each paper with the first month of its quarter, e.g. 2025-02 -> 2025-01
        unit = pd.Series(pd.PeriodIndex(unit, freq="Q").asfreq("M", how="start").strftime("%Y-%m"), index=unit.index)
    # pool units with fewer than min_cohort papers forward, as in section 5
    counts_ = unit.value_counts()
    cmap, cur, size = {}, None, 0
    for u in sorted(counts_.index):
        if cur is None or size >= min_cohort:
            cur, size = u, 0
        cmap[u] = cur
        size += counts_[u]
    coh = unit.map(cmap)
    c = cites[ok].astype(float)
    if kind == "pct":                                   # citation percentile within the cohort (section 5)
        tgt = c.groupby(coh).rank(pct=True, method="average") * 100
    else:                                               # log citations relative to the cohort mean
        lc = np.log10(1 + c)
        tgt = lc - lc.groupby(coh).transform("mean")
    return ok, tgt, coh


def run_variant(ok, tgt, coh, test_from=TEST_FROM, top_q=0.9, model=None):
    """Refit and score with a rebuilt target. Train = cohorts before test_from, test = the rest;
    'top' = papers at or above quantile top_q of their cohort."""
    Xv = F[ok].reset_index(drop=True)
    yv = tgt.to_numpy()
    cv = coh.to_numpy()
    trv, tev = cv < test_from, cv >= test_from
    topv = (tgt.groupby(coh).rank(pct=True) >= top_q).to_numpy()
    rho, auc = evaluate(model or hgb(), Xd=Xv, yd=yv, tr=trv, te=tev, top=topv)
    return rho, auc, int(tev.sum())


# sanity check: the default choices must reproduce section 5 exactly
# If this assertion failed, the variants below would differ from the reference for reasons other
# than the change being tested.
ok0, tgt0, coh0 = build_target()
assert np.allclose(tgt0.to_numpy(), y) and (coh0.to_numpy() == D["cohort"].to_numpy()).all()

# (name, changes to the target, changes to the split or the 'top' threshold).
VARIANTS = [("cohorts of at least 15 papers", dict(min_cohort=15), {}),
            ("cohorts of at least 50 papers", dict(min_cohort=50), {}),
            ("quarterly cohorts", dict(quarterly=True), {}),
            ("minimum age 6 months", dict(min_age=6), {}),
            ("minimum age 12 months", dict(min_age=12), {}),
            ("test from 2024-10", {}, dict(test_from="2024-10")),
            ("test from 2025-07", {}, dict(test_from="2025-07")),
            ("target: log citations vs. cohort mean", dict(kind="log"), {}),
            ("top 5% instead of top 10% (AUC)", {}, dict(top_q=0.95)),
            ("top 20% instead of top 10% (AUC)", {}, dict(top_q=0.80))]
rows_var = []
for name, tkw, rkw in VARIANTS:
    rho, auc, n = run_variant(*build_target(**tkw), **rkw)
    rows_var.append({"variant": name, "rho": rho, "AUC": auc, "test papers": n})
    record("8 Target and design", name, rho, auc, papers=n)
display(pd.DataFrame(rows_var).round(3))

### Test 9 · Are the linear associations of section 7 stable?

Four checks on the ridge regression of section 7: (a) the penalty strength varied by a factor 100 either way; (b) an elastic net, which can set weak effects exactly to zero, so it shows which effects survive a sparse model; (c) sign stability: in what fraction of the 300 bootstrap refits each effect keeps its sign; (d) collinearity: variance inflation factors (how much a coefficient's uncertainty is inflated because other features carry the same information; above about 5 is a warning), and the two first-author features refitted one at a time.

In [ ]:
# ── Test 9 · linear associations (Z, y, alpha, coef, boot from section 7) ──
TOP20 = df_assoc["feature"].head(20).tolist()           # the 20 largest effects of section 7
fi = [ASSOC.index(f) for f in TOP20]

# (a) penalty strength: alpha/100 ... alpha*100
# A larger penalty shrinks all coefficients towards zero; what matters is whether their order and
# sign hold.
pen = pd.DataFrame({f"alpha x {k:g}": Ridge(alpha=alpha * k).fit(Z, y).coef_[fi]
                    for k in (0.01, 0.1, 1, 10, 100)}, index=TOP20)

# (b) elastic net, mixing ridge and lasso penalties, both chosen by 5-fold cross-validation
enet = ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8, 1.0], cv=5, random_state=0, max_iter=20000).fit(Z, y)
pen["elastic net"] = enet.coef_[fi]

# (c) share of the bootstrap refits in which each coefficient has the same sign as the estimate
pen["same sign in bootstrap"] = (np.sign(boot[:, fi]) == np.sign(coef[fi])).mean(axis=0)
display(pen.round(2))
print(f"Elastic net: l1_ratio {enet.l1_ratio_}, {np.sum(enet.coef_ == 0)} of {len(ASSOC)} effects set to zero; "
      f"the 20 largest all kept: {bool(np.all(enet.coef_[fi] != 0))}")

# (d) collinearity: variance inflation factor of every feature = 1 / (1 - R^2) of regressing it on all the others
def vif(Zm):
    out = []
    for j in range(Zm.shape[1]):
        others = np.delete(Zm, j, axis=1)
        A = np.c_[np.ones(len(Zm)), others]
        resid = Zm[:, j] - A @ np.linalg.lstsq(A, Zm[:, j], rcond=None)[0]
        r2 = 1 - resid.var() / Zm[:, j].var() if Zm[:, j].var() > 0 else 0
        out.append(1 / max(1 - r2, 1e-9))
    return pd.Series(out, index=ASSOC)
# Values near 1 mean a feature is independent of the others; a VIF of 100 means its coefficient's
# variance is inflated 100-fold by collinearity.
VIF = vif(Z)
print("Largest variance inflation factors:")
display(VIF.sort_values(ascending=False).head(10).round(1))

# the two first-author features: correlation, and each one's coefficient without the other
j_prior, j_career = ASSOC.index("fa_log_prior_papers"), ASSOC.index("fa_career_years")
print(f"Correlation of 'earlier papers' and 'career length': {np.corrcoef(Z[:, j_prior], Z[:, j_career])[0, 1]:.2f}")
for drop_j, keep_j, keep_name in ((j_career, j_prior, "earlier papers"), (j_prior, j_career, "career length")):
    c_ = Ridge(alpha=alpha).fit(np.delete(Z, drop_j, axis=1), y).coef_
    k_ = keep_j - (keep_j > drop_j)                       # index shifts after deleting a column
    print(f"  {keep_name} alone: {c_[k_]:+.2f}   (with both: {coef[keep_j]:+.2f})")

### Test 10 · Self-citations

A citation counts as a self-citation when the citing paper shares at least one author (surname and first initial) with the cited paper. This needs a new ADS pull: the list of citing papers of every modelled paper and the authors of each citing paper (cached in `data/<freeze>/features_citing_authors.json`). Its citation counts can differ slightly from those of the topic pull (a citing paper can be counted by one pull and not the other), so both targets (all citations and non-self citations) are recomputed from this pull and compared like with like. Only the modelled papers are counted, and each ridge fit chooses its penalty by cross-validation on its own target, as everywhere else; `general/09_self_citations.py` repeats this with ORCID matching and gives the paper's numbers (its rule A reproduces this test exactly).

In [ ]:
# ── Test 10a · ADS pull: who cites each modelled paper, and the authors of those citing papers ──
sc = load_cache(SELF_JSON) or {"cited": {}, "citing_authors": {}}
# Step 1: for every modelled paper not yet cached, the bibcodes of the papers citing it, and its own
# authors. Citing lists change with time, so none is pulled after the freeze date: a paper the cache does not hold
# then is left out of this test.
need = [b for b in D["bibcode"] if b not in sc["cited"]]
if need and date.today().isoformat() > FREEZE and sc["cited"]:
    print(f"{len(need)} modelled papers have no citing list of the freeze: left out of test 10")
    need = []
if need:
    for d in bigquery(need, "bibcode,citation,author"):          # citation = bibcodes of the citing papers
        sc["cited"][d["bibcode"]] = {"citation": d.get("citation") or [], "author": d.get("author") or []}
    for b in need:
        sc["cited"].setdefault(b, {"citation": [], "author": []})
    save_cache(SELF_JSON, sc)
# Step 2: the authors of every citing paper not yet cached.
citing = sorted({c for v in sc["cited"].values() for c in v["citation"]} - set(sc["citing_authors"]))
for i in range(0, len(citing), 2000):                             # bigquery takes up to 2,000 bibcodes per page
    part = citing[i:i + 2000]
    for d in bigquery(part, "bibcode,author"):
        sc["citing_authors"][d["bibcode"]] = d.get("author") or []
    for b in part:
        sc["citing_authors"].setdefault(b, [])
    save_cache(SELF_JSON, sc)
print(f"{len(sc['cited'])} cited papers, {len(sc['citing_authors']):,} citing papers with their authors"
      f"{f' ({len(citing):,} fetched now)' if citing else ''}")

In [ ]:
# ── Test 10b · count self-citations and refit on non-self citations ──
def author_key(name):
    """'Müller, Anna B.' -> 'muller|a': surname without accents + first initial (the same person usually
    matches; namesakes with the same initial are counted as the same, which slightly overcounts self-citations)."""
    s = "".join(ch for ch in unicodedata.normalize("NFKD", name) if not unicodedata.combining(ch))
    surname, _, given = s.partition(",")
    return f"{surname.strip().lower()}|{given.strip()[:1].lower()}"


# Per modelled paper: number of citing papers, and how many of them share an author with it. (The cache can hold
# papers of earlier samples; they are left out.)
n_all, n_self = {}, {}
for b in D["bibcode"]:
    if b not in sc["cited"]:                     # no citing list of the freeze (Test 10a)
        continue
    v = sc["cited"][b]
    mine = {author_key(a) for a in v["author"]}
    cit = v["citation"]
    n_all[b] = len(cit)
    # self-citation: the citing paper shares at least one author key with this paper
    n_self[b] = sum(bool(mine & {author_key(a) for a in sc["citing_authors"].get(c, [])}) for c in cit)
# Self-citation counts and share per paper (share undefined for uncited papers).
self_df = pd.DataFrame({"all": pd.Series(n_all), "self": pd.Series(n_self)})
self_df["share"] = self_df["self"] / self_df["all"].where(self_df["all"] > 0)
tot = self_df[["all", "self"]].sum()
print(f"Self-citations: {int(tot['self']):,} of {int(tot['all']):,} citations ({tot['self'] / tot['all']:.1%}); "
      f"median share per paper {self_df['share'].median():.0%}")

# self-citation share by team size: are large teams' citations more self-made?
n_auth_self = pd.Series([len((meta.get(b) or {}).get("author") or []) for b in self_df.index], index=self_df.index)
team_bin = pd.cut(n_auth_self, [0, 5, 15, 50, 10_000], labels=["1-5 authors", "6-15", "16-50", "> 50"])
display(self_df.groupby(team_bin, observed=True)[["all", "self"]].sum()
        .assign(**{"self share": lambda t: (t["self"] / t["all"]).round(3)}))

# two targets from the same new pull: all citations, and citations by other authors only
cites_all = S["bibcode"].map(self_df["all"]).astype(float)
cites_ext = S["bibcode"].map(self_df["all"] - self_df["self"]).astype(float)
res_self = {}
for name, cts in (("all citations (new pull)", cites_all), ("without self-citations", cites_ext)):
    ok_, tgt_, coh_ = build_target(cites=cts)
    rho, auc, n = run_variant(ok_, tgt_, coh_)
    record("10 Self-citations", name, rho, auc, papers=n)
    # ridge effects on the same rows; the penalty chosen by cross-validation on this target, as in section 7
    XAs = assoc_coding(F[ok_], APJS[F[ok_].index])
    a_ = ridge().fit(XAs, tgt_.to_numpy())[-1].alpha_
    Zs = SimpleImputer(strategy="median").fit_transform(XAs)
    Zs = (Zs - Zs.mean(0)) / np.where(Zs.std(0) > 0, Zs.std(0), 1)
    res_self[name] = pd.Series(Ridge(alpha=a_).fit(Zs, tgt_.to_numpy()).coef_, index=ASSOC)
    print(f"{name}: test rho {rho:.3f}, AUC {auc:.3f}; ridge penalty {a_:.1f}")
# Side by side: each top-20 effect with all citations and without self-citations, and the change.
cmp_self = pd.DataFrame(res_self).loc[TOP20]
cmp_self["change"] = cmp_self.iloc[:, 1] - cmp_self.iloc[:, 0]
print("The 20 largest effects, with and without self-citations:")
display(cmp_self.round(2))

### Summary of the robustness tests

Every variant's test ρ, AUC, range (if any) and number of test papers, saved to `general/plots/robustness_summary.csv` (drawn in the paper's Extended Data Fig. 2 by `docs/paper/make_paper_figures.py`).

In [ ]:
# ── Summary table ──
# Every recorded variant: test, variant, rho, AUC, range (if any) and number of test papers.
df_rob = pd.DataFrame(ROBUST)
df_rob.round(4).to_csv(f"{PLOTS_DIR}/robustness_summary.csv", index=False)
display(df_rob.round(3))


### What the robustness tests show

Snapshot of 2026-10-19 (2,227 papers in the sample, 2,055 modelled; reference test ρ = 0.541 [0.486, 0.593], AUC 0.805).

- **Model settings hardly matter.** The 225 tree settings give ρ 0.46–0.57 (median 0.535); only very coarse leaves (at least 60 papers each) lower it, to about 0.46, because the trees can no longer single out rare features such as Nature/Science papers. Settings chosen without the test period give 0.567; random forests 0.44–0.55.
- **No leak.** With citation ranks shuffled within each month, ρ = −0.009 ± 0.060 (10 shuffles).
- **Publication information is not needed.** Without journal, refereed status, Letter, pages and references, ρ = 0.498.
- **Better data, better predictions.** Refereed papers only: 0.554; papers at least 6 or 12 months old: 0.592, 0.644. First authors whose name is not shared: 0.538, so incomplete author histories do not limit the model; without the author history: 0.516.
- **Target and design.** Other cohort sizes, quarterly cohorts and a later split give 0.54–0.55; an earlier split 0.459; log citations against the cohort mean 0.549.
- **Self-citations.** Without them ρ falls to 0.462: part of the skill, and much of the team-size effect, is a team citing itself.
- **Linear effects.** The largest ridge effects keep their sign in all 300 bootstrap refits and for a penalty 100 times weaker or 10 times stronger. Career length is negative only together with the number of earlier papers; alone it is +1.5.
